### Finnhub: Recommendation Trends

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, TimestampType
from pyspark.sql.functions import current_timestamp

recommendation_trends_schema = StructType([
    StructField("symbol", StringType()),
    StructField("period", StringType()),
    StructField("strongBuy", IntegerType()),
    StructField("buy", IntegerType()),
    StructField("hold", IntegerType()),
    StructField("sell", IntegerType()),
    StructField("strongSell", IntegerType()),
    StructField("_fetched_at", TimestampType()),
])

FAILED_LOADS = []


def full_refresh(df, table):
    """Replace table with df; if df is empty, keep the old table and record a failure."""
    n = df.count()
    if n == 0:
        FAILED_LOADS.append(table)
        print(f"[FAIL] {table}: new load has 0 rows; kept existing table")
        return
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: replaced with {n} rows")



In [0]:
VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data" 

df = spark.read.schema(recommendation_trends_schema).json(f"{VOLUME_PATH}/finnhub_recommendation_trends_*.json")
df.printSchema()
df.show()

In [0]:
df_bronze = df.withColumn("_ingested_at", current_timestamp())

df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bootcamp_students.oscr_bronze.finnhub_recommendation_trends")

spark.table("bootcamp_students.oscr_bronze.finnhub_recommendation_trends").show()


### Finnhub: Company News

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, LongType, TimestampType
from pyspark.sql.functions import current_timestamp

company_news_schema = StructType([
    StructField("category", StringType()),
    StructField("datetime", LongType()),
    StructField("headline", StringType()),
    StructField("id", LongType()),
    StructField("image", StringType()),
    StructField("related", StringType()),
    StructField("source", StringType()),
    StructField("summary", StringType()),
    StructField("url", StringType()),
    StructField("_fetched_at", TimestampType()),
])

In [0]:
VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data"

df = spark.read.schema(company_news_schema).json(f"{VOLUME_PATH}/finnhub_company_news_*.json")
df_bronze = df.withColumn("_ingested_at", current_timestamp())

In [0]:
full_refresh(df_bronze, "bootcamp_students.oscr_bronze.finnhub_company_news")

spark.table("bootcamp_students.oscr_bronze.finnhub_company_news").show()

### FMP: Analyst Estimates


In [0]:
from pyspark.sql.types import StructField, StructType, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.functions import current_timestamp

fmp_estimates_schema = StructType([
    StructField("symbol", StringType()),
    StructField("date", StringType()),
    StructField("revenueLow", DoubleType()),
    StructField("revenueHigh", DoubleType()),
    StructField("revenueAvg", DoubleType()),
    StructField("ebitdaLow", DoubleType()),
    StructField("ebitdaHigh", DoubleType()),
    StructField("ebitdaAvg", DoubleType()),
    StructField("ebitLow", DoubleType()),
    StructField("ebitHigh", DoubleType()),
    StructField("ebitAvg", DoubleType()),
    StructField("netIncomeLow", DoubleType()),
    StructField("netIncomeHigh", DoubleType()),
    StructField("netIncomeAvg", DoubleType()),
    StructField("sgaExpenseLow", DoubleType()),
    StructField("sgaExpenseHigh", DoubleType()),
    StructField("sgaExpenseAvg", DoubleType()),
    StructField("epsAvg", DoubleType()),
    StructField("epsHigh", DoubleType()),
    StructField("epsLow", DoubleType()),
    StructField("numAnalystsRevenue", IntegerType()),
    StructField("numAnalystsEps", IntegerType()),
    StructField("_fetched_at", TimestampType()),
])

In [0]:
VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data"

df = spark.read.schema(fmp_estimates_schema).json(f"{VOLUME_PATH}/fmp_analyst_estimates_*.json")
df_bronze = df.withColumn("_ingested_at", current_timestamp())

In [0]:
df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bootcamp_students.oscr_bronze.fmp_analyst_estimates")
spark.table("bootcamp_students.oscr_bronze.fmp_analyst_estimates").show()

# TMDB Reviews

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, DoubleType, ArrayType, LongType, TimestampType
from pyspark.sql.functions import explode, current_timestamp

review_item_schema = StructType([
    StructField("id", StringType()),
    StructField("author", StringType()),
    StructField("author_details", StructType([
        StructField("name", StringType()),
        StructField("username", StringType()),
        StructField("avatar_path", StringType()),
        StructField("rating", DoubleType()),
    ])),
    StructField("content", StringType()),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("url", StringType()),
])

tmdb_reviews_file_schema = StructType([
    StructField("id", LongType()),  # this is the SHOW's id, not a review id
    StructField("page", LongType()),
    StructField("results", ArrayType(review_item_schema)),
    StructField("total_pages", LongType()),
    StructField("total_results", LongType()),
    StructField("_fetched_at", TimestampType()),
])

In [0]:
VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data"

raw = spark.read.schema(tmdb_reviews_file_schema).json(f"{VOLUME_PATH}/tmdb_reviews_*.json")

df = raw.withColumnRenamed("id", "_show_id").withColumn("review", explode("results")).select("_show_id", "_fetched_at", "review.*")
df_bronze = df.withColumn("_ingested_at", current_timestamp())

full_refresh(df_bronze, "bootcamp_students.oscr_bronze.tmdb_reviews")
spark.table("bootcamp_students.oscr_bronze.tmdb_reviews").show(truncate=False)

In [0]:
registry_schema = StructType([
    StructField("show_id", LongType()),
    StructField("name", StringType()),
    StructField("platform", StringType()),
    StructField("_fetched_at", TimestampType()),
])

registry_df = spark.read.schema(registry_schema).json(f"{VOLUME_PATH}/tmdb_trending_show_registry.json")
registry_bronze = registry_df.withColumn("_ingested_at", current_timestamp())

full_refresh(registry_bronze, "bootcamp_students.oscr_bronze.tmdb_trending_shows")
spark.table("bootcamp_students.oscr_bronze.tmdb_trending_shows").show(truncate=False)

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, LongType, ArrayType, MapType, TimestampType
from pyspark.sql.functions import explode, current_timestamp

provider_schema = StructType([
    StructField("display_priorities", MapType(StringType(), LongType())),
    StructField("display_priority", LongType()),
    StructField("logo_path", StringType()),
    StructField("provider_name", StringType()),
    StructField("provider_id", LongType()),
])

watch_providers_file_schema = StructType([
    StructField("results", ArrayType(provider_schema)),
    StructField("_fetched_at", TimestampType()),
])

VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data"

raw = spark.read.schema(watch_providers_file_schema).json(f"{VOLUME_PATH}/tmdb_watch_providers.json")
df = raw.withColumn("provider", explode("results")).select("_fetched_at", "provider.*")
df_bronze = df.withColumn("_ingested_at", current_timestamp())

full_refresh(df_bronze, "bootcamp_students.oscr_bronze.tmdb_watch_providers")
spark.table("bootcamp_students.oscr_bronze.tmdb_watch_providers").show(truncate=False)

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, ArrayType, TimestampType
from pyspark.sql.functions import posexplode, current_timestamp, col

app_schema = StructType([
    StructField("artistName", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("releaseDate", StringType()),
    StructField("kind", StringType()),
    StructField("artworkUrl100", StringType()),
    StructField("genres", ArrayType(StringType())),
    StructField("url", StringType()),
])

feed_schema = StructType([
    StructField("feed", StructType([
        StructField("country", StringType()),
        StructField("results", ArrayType(app_schema)),
    ])),
    StructField("_fetched_at", TimestampType()),
])

VOLUME_PATH = "/Volumes/bootcamp_students/oscr_bronze/raw_landing/raw_data"

raw = spark.read.schema(feed_schema).json(f"{VOLUME_PATH}/appstore_top_free_100_*.json")

df = (
    raw
    .select(col("feed.country").alias("_storefront"), "_fetched_at", posexplode("feed.results").alias("_pos", "app"))
    .select("_storefront", "_fetched_at", (col("_pos") + 1).alias("_rank"), "app.*")
)
df_bronze = df.withColumn("_ingested_at", current_timestamp())

full_refresh(df_bronze, "bootcamp_students.oscr_bronze.appstore_top_free")
spark.table("bootcamp_students.oscr_bronze.appstore_top_free").show(truncate=False)

In [0]:
# Runs last so every other table still loads before the task fails.
if FAILED_LOADS:
    raise RuntimeError(f"Kept existing table(s) because the new load had 0 rows: {FAILED_LOADS}")
print("All full-refresh loads had rows.")